# Strategy ETFs: covered call funds vs the market

Companion notebook to the Medium post *Strategy ETFs: Why Covered Call Funds Are Winning Right Now, and When They Won't*,
and part 2 of the series that started with `theme_etfs.ipynb`. Change the funds in the **Configuration** cell and rerun
to compare any set of strategy ETFs.

> **Not investment advice.** An educational exploration only. Data comes from Yahoo Finance through the unofficial
> `yfinance` library and may be incomplete, delayed or wrong. Past performance says nothing reliable about the future.

| Chart | Question |
|---|---|
| Call payoff | What does a call option pay its buyer and its seller? |
| Volatility surface | How does implied volatility vary by strike and expiration, and what does fear do to it? *(illustrative)* |
| Covered call payoff | What does owning a stock and selling a call do to your outcome? |
| Comparison table | How does each fund implement the strategy, and what does it cost? |
| Strategy map | How much upside does each fund keep, and how much income does it pay? |
| Performance grid | How did each fund do against the S&P 500, Dow and Nasdaq, long term and short term? |

**How to run:** Runtime → Run all (Colab) or run cells top to bottom (Jupyter). Data is cached in `data/`, charts are saved
to `charts/` as HTML and PNG, tables as CSV.

## 0. Setup
Installs anything missing. `yfinance` is always upgraded, because Yahoo changes often and old versions break.

In [1]:
import importlib.util, subprocess, sys

REQUIRED = {"pandas": "pandas>=2.0", "numpy": "numpy>=1.26", "plotly": "plotly>=5.24",
            "nbformat": "nbformat>=4.2", "kaleido": "kaleido>=1.0"}
missing = [spec for mod, spec in REQUIRED.items() if importlib.util.find_spec(mod) is None]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "yfinance>=0.2.54", *missing])

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab" if IN_COLAB else "Running locally", "| packages ready")

Running in Colab | packages ready


## 1. Configuration
The only cell you need to edit to compare a different set of strategy ETFs.

In [2]:
STRATEGY = "covered call"

# The funds, in display order. Descriptions are yours (Yahoo has no field for them); numbers come from Yahoo.
# option_type groups funds on the strategy map: it largely decides how the income is taxed.
FUNDS = {
    "JEPI": {"approach": "Full overwrite via notes", "closest": "JEPQ",
             "mechanism": "Lower-volatility dividend stocks plus equity-linked notes that mimic selling "
                          "out-of-the-money S&P 500 index calls.",
             "option_type": "Equity-linked notes"},
    "JEPQ": {"approach": "Full overwrite via notes", "closest": "JEPI",
             "mechanism": "JEPI's equity-linked note approach applied to Nasdaq-100 stocks, where option "
                          "premiums are richer.",
             "option_type": "Equity-linked notes"},
    "KNG":  {"approach": "Partial overwrite", "closest": "DIVO",
             "mechanism": "Equal-weighted S&P 500 Dividend Aristocrats; sells calls on only part of each "
                          "position (about 15%).",
             "option_type": "Single-stock calls"},
    "DIVO": {"approach": "Tactical overwrite", "closest": "KNG",
             "mechanism": "Concentrated large-cap dividend growers; writes calls on individual stocks only "
                          "when premiums look attractive.",
             "option_type": "Single-stock calls"},
    "GPIX": {"approach": "Dynamic overwrite", "closest": "JEPI / KNG",
             "mechanism": "S&P 500 stocks; sells index calls on 25% to 75% of the portfolio, adjusted "
                          "month to month.",
             "option_type": "Index options"},
    "SPYI": {"approach": "Tax-aware high income", "closest": "JEPI",
             "mechanism": "Full S&P 500 replication; sells S&P 500 index calls taxed under Section 1256 "
                          "(60/40), with tax-loss harvesting.",
             "option_type": "Index options"},
    "OMAH": {"approach": "Fixed income target", "closest": "SPYI",
             "mechanism": "Berkshire Hathaway's top public holdings plus Berkshire; sells covered calls "
                          "to target 15% a year.",
             "option_type": "Single-stock calls"},
}

# Independent reference figures for the fact check against Yahoo (% values; source and date)
REFERENCE = {
    "JEPI": {"expense_ratio": 0.35, "source": "The Motley Fool, Oct 2026"},
    "JEPQ": {"expense_ratio": 0.35, "source": "ETF.com, May 2022"},
    "KNG":  {"expense_ratio": 0.74, "distribution_rate": 8.22, "source": "First Trust, Oct 30 2026"},
    "DIVO": {"expense_ratio": 0.56, "distribution_rate": 4.85, "source": "Amplify ETFs, Oct 31 2026"},
    "GPIX": {"expense_ratio": 0.29, "source": "ETF Database / Stock Analysis"},
    "SPYI": {"expense_ratio": 0.68, "distribution_rate": 11.43, "source": "NEOS fact sheet, Oct 31 2026"},
    "OMAH": {"expense_ratio": 0.95, "source": "ETF.com, Mar 2025"},
}

BENCHMARKS = {"SPY": "S&P 500", "DIA": "Dow Jones", "QQQ": "Nasdaq-100"}
VOL_INDEX = "^VIX"                 # shown under the performance grid; None to hide

# ---------- Performance grid ----------
LONG_PERIOD = "1Y"                 # left column:  "YTD", "1Y", "3Y", "5Y"
SHORT_PERIOD = "3M"                # right column: "1M", "3M", "6M"
START_VALUE = 10_000               # every line starts here

# ---------- Strategy map ----------
CAPTURE_PERIOD = "1Y"              # window for measuring up-capture
CAPTURE_BENCHMARK = "SPY"

# ---------- Theory charts (illustrative, not market data) ----------
THEORY = {
    "spot": 100, "strike": 103, "days": 30, "rate": 0.04,
    "vol_for_premium": 0.30,                         # volatility used to price the call in the payoff charts
    "calm":     {"level": 0.15, "skew": 0.10, "term": +0.03},   # term > 0: longer expiries cost more vol
    "stressed": {"level": 0.32, "skew": 0.20, "term": -0.08},   # term < 0: panic hits short expiries most
}

# ---------- Output ----------
DATA_DIR, OUT_DIR = "data", "charts"
REFRESH = False                    # True = download everything again
SAVE_PNG = True                    # also save PNGs (kaleido + Chrome; falls back to HTML only)

## 2. Style
One place for colors, fonts and sizes. Same pastel style as the theme notebook, different hues, so the two posts look like a series without implying the funds are related.

In [3]:
STYLE = {
    "palette": ["#A0E7E5", "#FBC4AB", "#CDB4DB", "#B9FBC0", "#FFC8DD", "#E9D8A6", "#A2D2FF",
                "#F1C0E8", "#C9F2C7", "#FFE3B3"],          # pastel, one per fund, in FUNDS order
    "benchmarks": {"SPY": ("#3B3B3B", "solid"), "DIA": ("#8C6D4F", "dash"), "QQQ": ("#3E5C8A", "dot")},
    "vol_index": "#B5838D",
    "option_symbols": {"Equity-linked notes": "circle", "Index options": "diamond", "Single-stock calls": "square"},
    "font": "Inter, Helvetica, Arial, sans-serif",
    "font_size": 12, "title_size": 20,
    "text": "#222222", "muted": "#777777", "grey": "#A0A0A0", "neutral": "#D9D9D9",
    "template": "plotly_white",
    "dark_factor": 0.55,                    # lines and labels are this much darker than their pastel fill
    "png_scale": 2,
}

## 3. Helpers

In [4]:
import math, os, time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf

TICKERS = list(FUNDS)
COLOR = {f: STYLE["palette"][i % len(STYLE["palette"])] for i, f in enumerate(TICKERS)}
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
PERIOD_NAME = {"1M": "1 month", "3M": "3 months", "6M": "6 months", "YTD": "year to date",
               "1Y": "1 year", "3Y": "3 years", "5Y": "5 years"}

def darker(hex_color, f=None):
    f = STYLE["dark_factor"] if f is None else f
    h = hex_color.lstrip("#")
    return "rgb({},{},{})".format(*(int(int(h[i:i + 2], 16) * f) for i in (0, 2, 4)))

def rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    return "rgba({},{},{},{})".format(*(int(h[i:i + 2], 16) for i in (0, 2, 4)), alpha)

def styled(fig, title, subtitle=None, **layout):
    text = title + (f"<br><sup>{subtitle}</sup>" if subtitle else "")
    fig.update_layout(template=STYLE["template"],
                      font=dict(family=STYLE["font"], size=STYLE["font_size"], color=STYLE["text"]),
                      title=dict(text=text, x=0.01, font=dict(size=STYLE["title_size"])), **layout)
    return fig

_png_ok = SAVE_PNG
def save(fig, name):
    global _png_ok
    fig.write_html(os.path.join(OUT_DIR, f"{name}.html"), include_plotlyjs="cdn")
    if _png_ok:
        try:
            fig.write_image(os.path.join(OUT_DIR, f"{name}.png"), scale=STYLE["png_scale"])
        except Exception:
            try:
                import kaleido; kaleido.get_chrome_sync()
                fig.write_image(os.path.join(OUT_DIR, f"{name}.png"), scale=STYLE["png_scale"])
            except Exception as e:
                _png_ok = False
                print(f"PNG export unavailable ({type(e).__name__}); saving HTML only.")
    fig.show()

def period_start(index, end, p):
    if p == "YTD":
        return index[index.year < end.year].max()
    n, unit = int(p[:-1]), p[-1]
    return end - (pd.DateOffset(months=n) if unit == "M" else pd.DateOffset(years=n))

def growth(series, start, end, base):
    s = series.dropna()
    s = s[(s.index >= start) & (s.index <= end)]
    if s.empty or (s.index[0] - start).days > 7:
        return None                                   # fund too young for this window
    return base * s / s.iloc[0]

def retry(fn, tries=3, wait=2):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            if i == tries - 1:
                print(f"   failed: {e}")
                return None
            time.sleep(wait * (i + 1))

def spread(values, gap):
    order = np.argsort(values)
    pos = np.array(values, dtype=float)[order]
    for i in range(1, len(pos)):
        pos[i] = max(pos[i], pos[i - 1] + gap)
    out = np.empty_like(pos)
    out[order] = pos - max(0, (pos.max() - np.max(values)) / 2)
    return out

def bs_call(S, K, T, r, vol):
    # Black-Scholes price of a European call
    from math import erf, exp, log, sqrt
    N = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
    d1 = (log(S / K) + (r + vol ** 2 / 2) * T) / (vol * sqrt(T))
    return S * N(d1) - K * exp(-r * T) * N(d1 - vol * sqrt(T))

## 4. Data from Yahoo Finance
Fund facts and dividend-adjusted prices for the funds, the benchmarks and the VIX, cached in `data/`. The check table compares Yahoo with the independent figures in `REFERENCE`.

In [5]:
def cached(name, builder, **read_kw):
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path) and not REFRESH:
        return pd.read_csv(path, **read_kw)
    df = builder()
    df.to_csv(path, index=read_kw.get("index_col") is not None)
    return pd.read_csv(path, **read_kw)

def fetch_info():
    rows = []
    for f in TICKERS + list(BENCHMARKS):
        info = retry(lambda: yf.Ticker(f).info) or {}
        inception = info.get("fundInceptionDate")
        rows.append({"ticker": f, "name": info.get("longName") or info.get("shortName") or f,
                     "issuer": info.get("fundFamily", ""),
                     "expense_ratio": info.get("netExpenseRatio", np.nan),          # in %
                     "yield": (info.get("yield") or np.nan) * 100,                    # trailing 12 months, %
                     "total_assets": info.get("totalAssets", np.nan),
                     "inception": pd.to_datetime(inception, unit="s").date() if inception else None})
        time.sleep(0.5)
    return pd.DataFrame(rows)

START = (pd.Timestamp.today() - pd.DateOffset(years=6)).date()
def fetch_prices():
    tickers = TICKERS + list(BENCHMARKS) + ([VOL_INDEX] if VOL_INDEX else [])
    df = retry(lambda: yf.download(tickers, start=START, auto_adjust=True, progress=False)["Close"])
    return df

INFO = cached("strategy_info.csv", fetch_info).set_index("ticker")
PX = cached("strategy_prices.csv", fetch_prices, index_col=0, parse_dates=True)
END = PX[TICKERS].dropna(how="all").index.max()
first_price = {t: PX[t].first_valid_index() for t in PX}
def launched(t):
    i = INFO.loc[t, "inception"]
    if pd.notna(i) and i:
        return pd.to_datetime(i).strftime("%b %Y")
    fp = first_price[t]                                      # no inception date from Yahoo: use price history
    return f"before {fp:%b %Y}" if (fp - pd.Timestamp(START)).days < 10 else fp.strftime("%b %Y")

# ---------- Fact check: Yahoo vs independent sources ----------
chk = pd.DataFrame(index=TICKERS)
chk["expense_ratio_yahoo"] = INFO["expense_ratio"]
chk["expense_ratio_ref"] = [REFERENCE.get(t, {}).get("expense_ratio", np.nan) for t in TICKERS]
chk["yield_yahoo"] = INFO["yield"]
chk["distribution_rate_ref"] = [REFERENCE.get(t, {}).get("distribution_rate", np.nan) for t in TICKERS]
chk["first_price"] = [first_price[t].date() for t in TICKERS]
chk["source"] = [REFERENCE.get(t, {}).get("source", "") for t in TICKERS]
flags = []
for t, r in chk.iterrows():
    if pd.notna(r["expense_ratio_ref"]) and (pd.isna(r["expense_ratio_yahoo"]) or abs(r["expense_ratio_yahoo"] - r["expense_ratio_ref"]) > 0.02):
        flags.append(f"{t}: expense ratio Yahoo {r['expense_ratio_yahoo']}% vs reference {r['expense_ratio_ref']}%")
    if pd.notna(r["distribution_rate_ref"]) and pd.notna(r["yield_yahoo"]) and abs(r["yield_yahoo"] - r["distribution_rate_ref"]) > 2:
        flags.append(f"{t}: trailing yield {r['yield_yahoo']:.1f}% vs distribution rate {r['distribution_rate_ref']}% "
                     "(different measures; check which one to quote)")
print(f"Prices through {END:%b %d, %Y}")
display(chk.style.format({c: "{:.2f}%" for c in ["expense_ratio_yahoo", "expense_ratio_ref", "yield_yahoo",
                                                 "distribution_rate_ref"]}, na_rep="n/a"))
print("Flags:", *(flags or ["none"]), sep="\n  ")

Prices through Nov 17, 2026


,expense_ratio_yahoo,expense_ratio_ref,yield_yahoo,distribution_rate_ref,first_price,source
JEPI,0.35%,0.35%,7.93%,n/a,2020-11-17,"The Motley Fool, Oct 2026"
JEPQ,0.35%,0.35%,11.96%,n/a,2022-05-04,"ETF.com, May 2022"
KNG,0.74%,0.74%,5.98%,8.22%,2020-11-17,"First Trust, Oct 30 2026"
DIVO,0.56%,0.56%,1.95%,4.85%,2020-11-17,"Amplify ETFs, Oct 31 2026"
GPIX,0.29%,0.29%,0.95%,n/a,2023-10-27,ETF Database / Stock Analysis
SPYI,0.68%,0.68%,1.43%,11.43%,2022-08-30,"NEOS fact sheet, Oct 31 2026"
OMAH,0.98%,0.95%,6.04%,n/a,2025-03-05,"ETF.com, Mar 2025"


Flags:
  KNG: trailing yield 6.0% vs distribution rate 8.22% (different measures; check which one to quote)
  DIVO: trailing yield 1.9% vs distribution rate 4.85% (different measures; check which one to quote)
  SPYI: trailing yield 1.4% vs distribution rate 11.43% (different measures; check which one to quote)
  OMAH: expense ratio Yahoo 0.98% vs reference 0.95%


## 5. Theory charts (illustrative)
These use a standard pricing model with made-up inputs from `THEORY`, not market data.

In [6]:
S0, K, T, r = THEORY["spot"], THEORY["strike"], THEORY["days"] / 365, THEORY["rate"]
PREMIUM = bs_call(S0, K, T, r, THEORY["vol_for_premium"])
S = np.linspace(S0 * 0.75, S0 * 1.25, 201)
BUYER, SELLER = darker(STYLE["palette"][0], 0.6), darker(STYLE["palette"][1], 0.6)

def payoff_axes(fig, title, subtitle):
    fig.add_hline(y=0, line=dict(color=STYLE["grey"], width=1))
    fig.add_vline(x=K, line=dict(color=STYLE["grey"], width=1, dash="dot"),
                  annotation_text=f"strike {K}", annotation_position="bottom right")
    styled(fig, title, subtitle, width=900, height=520, margin=dict(l=60, r=30, t=90, b=60),
           xaxis_title="Stock price at expiration", yaxis_title="Profit or loss per share",
           legend=dict(orientation="h", y=-0.18))

def call_payoff():
    buyer = np.maximum(S - K, 0) - PREMIUM
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=S, y=buyer, name="Call buyer", line=dict(color=BUYER, width=3)))
    fig.add_trace(go.Scatter(x=S, y=-buyer, name="Call seller", line=dict(color=SELLER, width=3)))
    be = K + PREMIUM
    fig.add_annotation(x=be, y=0, text=f"break-even {be:.1f}", showarrow=True, ay=-40, font=dict(color=BUYER))
    fig.add_annotation(x=S[5], y=-PREMIUM, text=f"buyer's loss is limited to the premium ({PREMIUM:.2f})",
                       showarrow=False, xanchor="left", yshift=-12, font=dict(color=BUYER))
    fig.add_annotation(x=S[5], y=PREMIUM, text="seller keeps the premium", showarrow=False, xanchor="left",
                       yshift=12, font=dict(color=SELLER))
    payoff_axes(fig, "A call option at expiration: the buyer's gain is the seller's loss",
                f"Stock at {S0}, strike {K}, {THEORY['days']} days, premium {PREMIUM:.2f} (illustrative)")
    save(fig, "theory1_call_payoff")

def iv_surface(p):
    m = np.linspace(0.80, 1.20, 41)                  # strike as a fraction of today's price
    t = np.linspace(1 / 12, 1.0, 24)                 # years to expiration
    M, TT = np.meshgrid(m, t)
    term = p["level"] + p["term"] * np.sqrt(TT)                       # term structure
    skew = p["skew"] * (1 - M) / np.sqrt(TT) * 0.35                  # puts (low strikes) cost more, most at short T
    smile = 0.25 * (1 - M) ** 2 / np.sqrt(TT)
    return m * 100, t * 12, np.clip(term + skew + smile, 0.05, None) * 100

def vol_surfaces():
    # Both surfaces in one scene, so the lift from calm to stressed is visible directly
    fig = go.Figure()
    zmax = 0
    for key, label, color in [("calm", "Calm market", STYLE["palette"][6]), ("stressed", "Stressed market", STYLE["palette"][1])]:
        x, y, z = iv_surface(THEORY[key])
        zmax = max(zmax, z.max())
        fig.add_trace(go.Surface(x=x, y=y, z=z, name=label, showlegend=True, showscale=False, opacity=0.93,
                                 colorscale=[[0, color], [1, darker(color, 0.75)]],
                                 contours=dict(x=dict(show=True, color="white", width=1, start=80, end=120, size=5),
                                               y=dict(show=True, color="white", width=1, start=1, end=12, size=1)),
                                 lighting=dict(ambient=0.75, diffuse=0.5, specular=0.05),
                                 hovertemplate=f"{label}<br>strike %{{x:.0f}}%<br>%{{y:.0f}} months<br>IV %{{z:.1f}}%<extra></extra>"))
    fig.update_layout(scene=dict(
        xaxis=dict(title="Strike, % of today's price", autorange="reversed"),
        yaxis=dict(title="Months to expiry"), zaxis=dict(title="Implied volatility %", range=[0, zmax * 1.1]),
        camera=dict(eye=dict(x=1.5, y=-1.35, z=0.6), center=dict(z=-0.22)), aspectratio=dict(x=1.3, y=1.0, z=0.85)),
        legend=dict(x=0.02, y=0.85, font=dict(size=13)))
    styled(fig, "The volatility surface: what fear does to option prices",
           "Illustrative. In a selloff the whole surface lifts, low strikes (protection) rise most, and short expiries jump most",
           width=1100, height=720, margin=dict(l=10, r=10, t=100, b=10))
    save(fig, "theory2_vol_surface")

def covered_call_payoff():
    stock = S - S0
    cc = np.minimum(S, K) - S0 + PREMIUM
    STOCK, CC = STYLE["grey"], darker(STYLE["palette"][3], 0.5)
    fig = go.Figure()
    above = S >= K + PREMIUM                       # beyond break-even the stock alone does better
    fig.add_trace(go.Scatter(x=S[above], y=stock[above], line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=S[above], y=cc[above], fill="tonexty", fillcolor=rgba(STYLE["palette"][1], 0.55),
                             line=dict(width=0), name="Upside given away"))
    below = S <= K
    fig.add_trace(go.Scatter(x=S[below], y=stock[below], line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=S[below], y=cc[below], fill="tonexty",
                             fillcolor=rgba(STYLE["palette"][3], 0.7), line=dict(width=0), name="Premium cushion"))
    fig.add_trace(go.Scatter(x=S, y=stock, name="Own the stock", line=dict(color=STOCK, width=2.5, dash="dash")))
    fig.add_trace(go.Scatter(x=S, y=cc, name="Covered call (stock + sold call)", line=dict(color=CC, width=3.5)))
    payoff_axes(fig, "A covered call: a cushion on the way down, a cap on the way up",
                f"Bought at {S0}, sold a {THEORY['days']}-day call with strike {K} for {PREMIUM:.2f} (illustrative)")
    save(fig, "theory3_covered_call")

call_payoff()
vol_surfaces()
covered_call_payoff()

/usr/local/lib/python3.13/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




PNG export unavailable (ValueError); saving HTML only.


## 6. The funds compared

In [7]:
def total_return(t, p):
    g = growth(PX[t], period_start(PX.index, END, p), END, 1.0)
    return g.iloc[-1] - 1 if g is not None else np.nan

def comparison_table():
    rows = []
    for t in TICKERS:
        cfg, inf = FUNDS[t], INFO.loc[t]
        rows.append([f"<b>{t}</b>", inf["name"], cfg["approach"], cfg["mechanism"], cfg["option_type"],
                     f"{inf['yield']:.1f}%" if pd.notna(inf["yield"]) else "n/a",
                     f"{inf['expense_ratio']:.2f}%" if pd.notna(inf["expense_ratio"]) else "n/a",
                     launched(t), cfg["closest"]])
    cols = ["Ticker", "Fund", "Approach", "Strategy and options mechanism", "Option type",
            "Yield (TTM)", "Expense ratio", "Launched", "Closest match"]
    table = pd.DataFrame(rows, columns=cols)
    fills = [[COLOR[t] for t in TICKERS]] + [[rgba(COLOR[t], 0.28) for t in TICKERS]] * (len(cols) - 1)
    fig = go.Figure(go.Table(
        columnwidth=[60, 190, 120, 330, 120, 80, 80, 80, 90],
        header=dict(values=[f"<b>{c}</b>" for c in cols], fill_color="#F4F4F4", align="left",
                    font=dict(size=12, color=STYLE["text"]), line_color="white", height=34),
        cells=dict(values=[table[c] for c in cols], fill_color=fills, align="left", height=48,
                   font=dict(size=11, color=STYLE["text"]), line_color="white")))
    styled(fig, f"Seven {STRATEGY} ETFs, seven ways to sell upside",
           f"Yield and expense ratio from Yahoo Finance, as of {END:%b %d, %Y}; descriptions from fund documents",
           width=1350, height=120 + 54 * len(TICKERS), margin=dict(l=10, r=10, t=80, b=10))
    save(fig, "chart1_comparison_table")
    table.assign(Ticker=TICKERS).to_csv(os.path.join(OUT_DIR, "comparison_table.csv"), index=False)

def capture(t, start, bench):
    d = PX.loc[PX.index >= start, [t, bench]].pct_change().dropna()
    up, down = d[bench] > 0, d[bench] < 0
    return d.loc[up, t].mean() / d.loc[up, bench].mean(), d.loc[down, t].mean() / d.loc[down, bench].mean()

def strategy_map():
    start = period_start(PX.index, END, CAPTURE_PERIOD)
    pts = []
    for t in TICKERS:
        if growth(PX[t], start, END, 1) is None:
            print(f"{t}: too young for a {PERIOD_NAME[CAPTURE_PERIOD]} capture window, left off the map")
            continue
        upc, dnc = capture(t, start, CAPTURE_BENCHMARK)
        pts.append({"t": t, "up": upc, "down": dnc, "yld": INFO.loc[t, "yield"], "er": INFO.loc[t, "expense_ratio"]})
    pts = pd.DataFrame(pts).sort_values("er", ascending=False)     # big markers first, small ones on top
    def label_position(r):
        # put the label below the marker when a neighbor sits just above it
        crowded = ((abs(pts["up"] - r.up) < 0.04) & (pts["yld"] > r.yld) & (pts["yld"] - r.yld < 1.5)).any()
        return "bottom center" if crowded else "top center"
    fig = go.Figure()
    for r in pts.itertuples():
        fig.add_trace(go.Scatter(
            x=[r.up], y=[r.yld], mode="markers+text", text=[f"<b>{r.t}</b>"], textposition=label_position(r),
            textfont=dict(size=13, color=darker(COLOR[r.t], 0.45)), showlegend=False,
            marker=dict(size=14 + 28 * (r.er or 0), color=COLOR[r.t], opacity=0.9, symbol=STYLE["option_symbols"][FUNDS[r.t]["option_type"]],
                        line=dict(color=darker(COLOR[r.t]), width=2)),
            hovertemplate=(f"<b>{r.t}</b><br>up-capture %{{x:.0%}}<br>down-capture {r.down:.0%}<br>"
                           f"yield %{{y:.1f}}%<br>expense ratio {r.er:.2f}%<extra></extra>")))
    by = INFO.loc[CAPTURE_BENCHMARK, "yield"]
    fig.add_trace(go.Scatter(x=[1.0], y=[by], mode="markers+text", text=[f"{CAPTURE_BENCHMARK} ({BENCHMARKS[CAPTURE_BENCHMARK]})"],
                             textposition="middle left", marker=dict(size=12, color=STYLE["benchmarks"]["SPY"][0], symbol="x"),
                             showlegend=False, hoverinfo="skip"))
    for name, sym in STYLE["option_symbols"].items():          # legend for option types
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=name,
                                 marker=dict(size=12, symbol=sym, color="white", line=dict(color=STYLE["muted"], width=2))))
    ymax = np.nanmax(pts["yld"].tolist() + [by]) * 1.2
    fig.add_annotation(xref="paper", yref="paper", x=0.98, y=0.97, xanchor="right", showarrow=False,
                       text="<i>High yield and full upside: the empty corner</i>", font=dict(color=STYLE["muted"]))
    fig.add_annotation(xref="paper", yref="paper", x=0, y=-0.25, xanchor="left", showarrow=False,
                       text="Bubble size = expense ratio. Up-capture = the fund's average return on the benchmark's up days, "
                            "divided by the benchmark's.", font=dict(size=11, color=STYLE["muted"]))
    styled(fig, "The strategy map: income vs the upside you keep",
           f"Up-capture vs {CAPTURE_BENCHMARK} over {PERIOD_NAME[CAPTURE_PERIOD]} to {END:%b %d, %Y} · trailing 12-month yield from Yahoo",
           width=1000, height=680, margin=dict(l=70, r=30, t=90, b=140),
           xaxis=dict(title="Up-capture: share of the S&P 500's up days kept", tickformat=".0%"),
           yaxis=dict(title="Trailing 12-month yield", ticksuffix="%", range=[0, ymax]),
           legend=dict(title="Option type: ", orientation="h", y=-0.15, x=0, xanchor="left", yanchor="top"))
    save(fig, "chart2_strategy_map")
    pts.to_csv(os.path.join(OUT_DIR, "strategy_map.csv"), index=False)

comparison_table()
strategy_map()

/tmp/ipykernel_4081/4128189793.py:30: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

/tmp/ipykernel_4081/4128189793.py:30: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

/tmp/ipykernel_4081/4128189793.py:30: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

/tmp/ipykernel_4081/4128189793.py:30: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed 

## 7. Performance vs the major indices
Each row is one fund (shaded area) against SPY, DIA and QQQ (lines), all as growth of $10,000 with distributions reinvested. Left: long term; right: short term. A panel stays blank when the fund is too young for the window.

In [8]:
def performance_grid():
    periods = [LONG_PERIOD, SHORT_PERIOD]
    starts = {p: period_start(PX.index, END, p) for p in periods}
    vol_row = 1 if VOL_INDEX else 0
    n_rows = len(TICKERS) + vol_row
    titles = []
    for t in TICKERS:
        for p in periods:
            g = growth(PX[t], starts[p], END, START_VALUE)
            titles.append(f"<b>{t}</b> · {PERIOD_NAME[p]}" + (f": {g.iloc[-1] / START_VALUE - 1:+.1%}" if g is not None else ""))
    if VOL_INDEX:
        titles += [f"VIX · {PERIOD_NAME[p]}" for p in periods]
    fig = make_subplots(rows=n_rows, cols=2, shared_xaxes="columns", horizontal_spacing=0.17, vertical_spacing=0.025,
                        subplot_titles=titles, row_heights=[1] * len(TICKERS) + ([0.7] if VOL_INDEX else []))
    for a in fig.layout.annotations:
        a.update(font_size=12)
    pct = lambda v: f"{v / START_VALUE - 1:+.1%}"
    shown = set()
    for row, t in enumerate(TICKERS, 1):
        for col, p in enumerate(periods, 1):
            n = (row - 1) * 2 + col
            yref, xname = ("y" if n == 1 else f"y{n}"), ("xaxis" if n == 1 else f"xaxis{n}")
            g = growth(PX[t], starts[p], END, START_VALUE)
            if g is None:
                fig.add_annotation(x=0.5, y=0.5, xref=f"x{n} domain" if n > 1 else "x domain",
                                   yref=f"y{n} domain" if n > 1 else "y domain", showarrow=False,
                                   text=f"Not enough history (launched {launched(t)})",
                                   font=dict(color=STYLE["muted"]))
                fig.update_yaxes(showticklabels=False, row=row, col=col)
                continue
            dark = darker(COLOR[t])
            fig.add_trace(go.Scatter(x=g.index, y=[START_VALUE] * len(g), line=dict(width=0), hoverinfo="skip",
                                     showlegend=False), row=row, col=col)
            fig.add_trace(go.Scatter(x=g.index, y=g, fill="tonexty", fillcolor=COLOR[t], showlegend=False,
                                     line=dict(color=dark, width=2.2),
                                     hovertemplate=f"<b>{t}</b> %{{x|%b %d, %Y}}<br>$%{{y:,.0f}}<extra></extra>"),
                          row=row, col=col)
            names = [(f"<b>{t}</b> {pct(g.iloc[-1])}", g.iloc[-1], dark)]
            for b, label in BENCHMARKS.items():
                bg = growth(PX[b], starts[p], END, START_VALUE)
                c, dash = STYLE["benchmarks"][b]
                fig.add_trace(go.Scatter(x=bg.index, y=bg, name=f"{b} ({label})", legendgroup=b,
                                         showlegend=b not in shown, line=dict(color=c, width=1.4, dash=dash),
                                         hovertemplate=f"{b} %{{x|%b %d, %Y}}<br>$%{{y:,.0f}}<extra></extra>"),
                              row=row, col=col)
                shown.add(b)
                names.append((f"{b} {pct(bg.iloc[-1])}", bg.iloc[-1], c))
            fig.add_hline(y=START_VALUE, line=dict(color=STYLE["grey"], width=0.8, dash="dot"), row=row, col=col)
            fig.update_yaxes(tickprefix="$", tickformat="~s", row=row, col=col)
            vals = [v for _, v, _ in names]
            gap = ((max(vals) - min(vals)) or START_VALUE * 0.02) * 0.17
            x_label = fig.layout[xname].domain[1] + 0.004
            for (text, _, c), yv in zip(names, spread(vals, gap)):
                fig.add_annotation(x=x_label, xref="paper", y=yv, yref=yref, text=text, showarrow=False,
                                   xanchor="left", font=dict(size=10, color=c))
    if VOL_INDEX:
        for col, p in enumerate(periods, 1):
            v = PX[VOL_INDEX].loc[PX.index >= starts[p]].dropna()
            fig.add_trace(go.Scatter(x=v.index, y=v, fill="tozeroy", fillcolor=rgba(STYLE["vol_index"], 0.25),
                                     line=dict(color=STYLE["vol_index"], width=1.5), showlegend=False,
                                     hovertemplate="VIX %{x|%b %d, %Y}: %{y:.1f}<extra></extra>"), row=n_rows, col=col)
            fig.add_hline(y=20, line=dict(color=STYLE["grey"], width=0.8, dash="dot"), row=n_rows, col=col)
    styled(fig, f"{STRATEGY.capitalize()} ETFs vs the S&P 500, Dow and Nasdaq: growth of ${START_VALUE:,.0f}",
           f"left: {PERIOD_NAME[LONG_PERIOD]} from {starts[LONG_PERIOD]:%b %d, %Y} · right: {PERIOD_NAME[SHORT_PERIOD]} "
           f"from {starts[SHORT_PERIOD]:%b %d, %Y} · to {END:%b %d, %Y} · total return, distributions reinvested",
           height=260 * n_rows + 150, width=1400, margin=dict(l=60, r=150, t=150, b=40), hovermode="closest",
           legend=dict(orientation="h", y=1.0, yref="container", yanchor="top", x=0.99, xanchor="right",
                       entrywidth=150))
    save(fig, "chart3_performance_grid")

def returns_table():
    cols = ["1M", "3M", "6M", "YTD", "1Y", "3Y", "5Y"]
    tbl = pd.DataFrame({p: [total_return(t, p) for t in TICKERS + list(BENCHMARKS)] for p in cols},
                       index=TICKERS + list(BENCHMARKS))
    tbl.to_csv(os.path.join(OUT_DIR, "total_returns.csv"))
    display(tbl.style.format("{:+.1%}", na_rep="n/a").set_caption(f"Total returns to {END:%b %d, %Y}"))

performance_grid()
returns_table()
print(f"\nAll charts and tables saved to '{OUT_DIR}/'.")

,1M,3M,6M,YTD,1Y,3Y,5Y
JEPI,+0.1%,+0.1%,+5.2%,+6.0%,+9.1%,+32.6%,+41.4%
JEPQ,-8.0%,-7.7%,-3.3%,+3.0%,+6.3%,+53.6%,n/a
KNG,+5.7%,+4.3%,+12.9%,+15.0%,+18.0%,+37.4%,+33.4%
DIVO,-2.7%,-3.2%,+3.9%,+8.3%,+10.9%,+54.8%,+60.8%
GPIX,-4.0%,-3.7%,+1.0%,+9.0%,+12.1%,+62.4%,n/a
SPYI,-3.6%,-3.0%,+1.3%,+7.5%,+11.1%,+53.7%,n/a
OMAH,+2.2%,+3.1%,+3.7%,+10.6%,+12.7%,n/a,n/a
SPY,-4.5%,-5.0%,-0.4%,+8.2%,+11.2%,+68.4%,+67.2%
DIA,-2.0%,-4.5%,+3.0%,+7.0%,+10.6%,+52.3%,+54.1%
QQQ,-8.9%,-10.2%,-7.0%,+7.0%,+9.0%,+72.5%,+69.7%



All charts and tables saved to 'charts/'.
